# Spatial Analysis of Chicago Traffic Crashes
**Author:** [Chris Kucewicz](https://www.linkedin.com/in/chriskucewicz/)

> Mapping crash density, severity patterns, and neighborhood-level risk across Chicago using interactive visualizations.

This notebook extends the main analysis in `notebook.ipynb` with a geographic lens. Using latitude/longitude coordinates from the raw crash dataset, we build three interactive maps:

1. **Heatmap** — Overall crash density across Chicago
2. **Severity Point Map** — Serious vs. non-serious crashes as colored markers
3. **Choropleth** — Serious injury rate by Chicago community area

Maps are exported as standalone HTML files to `docs/maps/` for embedding in the project GitHub Pages site.

---

## 1. Setup & Data Loading

In [1]:
# Install dependencies if needed
# !pip install folium geopandas requests -q

In [2]:
import os
import zipfile
import json
import requests

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

import folium
from folium.plugins import HeatMap, MarkerCluster

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

# Output directory for saved maps
os.makedirs('docs/maps', exist_ok=True)

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [3]:
# ── Load data from Kaggle (same setup as existing notebooks) ──────────────────

# Step 1: Install Kaggle API
# !pip install -q kaggle

# Step 2: Setup credentials
# !cp kaggle.json /root/.kaggle/
# !chmod 600 /root/.kaggle/kaggle.json

# Step 3: Download
# dataset_name = "ckucewicz/chicago-traffic-data"
# !kaggle datasets download -d {dataset_name}

# Step 4: Extract
# zip_file = dataset_name.split('/')[-1] + ".zip"
# with zipfile.ZipFile(zip_file, 'r') as zip_ref:
#     zip_ref.extractall('.')

# Step 5: Load the raw crashes CSV
KEEP_COLS = [
    'CRASH_RECORD_ID',
    'MOST_SEVERE_INJURY',
    'LATITUDE',
    'LONGITUDE',
    'CRASH_DATE',
    'CRASH_MONTH',
    'PRIM_CONTRIBUTORY_CAUSE',
]

crashes_raw = pd.read_csv(
    'chicago_traffic_data/traffic_crashes.csv',
    usecols=KEEP_COLS,
    low_memory=False
)

# Lowercase columns
crashes_raw.columns = crashes_raw.columns.str.lower()

# Filter to 2021 onward
crashes_raw['crash_date'] = pd.to_datetime(crashes_raw['crash_date'])
crashes_raw = crashes_raw[crashes_raw['crash_date'].dt.year >= 2021]

print(f'Records from 2021 onward: {len(crashes_raw):,}')
crashes_raw.head(3)

Records from 2021 onward: 434,720


,crash_record_id,crash_date,prim_contributory_cause,most_severe_injury,crash_month,latitude,longitude
0,6c1659069e9c6285a650e70d6f9b574ed5f64c12888479...,2023-08-18 12:50:00,FOLLOWING TOO CLOSELY,NONINCAPACITATING INJURY,8,NaN,NaN
1,5f54a59fcb087b12ae5b1acff96a3caf4f2d37e79f8db4...,2023-07-29 14:45:00,FAILING TO REDUCE SPEED TO AVOID CRASH,NO INDICATION OF INJURY,7,41.854120,-87.665902
2,61fcb8c1eb522a6469b460e2134df3d15f82e81fd93e9c...,2023-08-18 17:58:00,FAILING TO REDUCE SPEED TO AVOID CRASH,NONINCAPACITATING INJURY,8,41.942976,-87.761883


## 2. Data Preparation

In [4]:
# Standardize column names and string values
crashes_raw.columns = crashes_raw.columns.str.lower()
for col in crashes_raw.select_dtypes(include='object').columns:
    crashes_raw[col] = crashes_raw[col].str.lower().str.strip()

# Drop rows missing coordinates
crashes_geo = crashes_raw.dropna(subset=['latitude', 'longitude']).copy()
print(f'Records with coordinates: {len(crashes_geo):,} '
      f'({len(crashes_geo)/len(crashes_raw)*100:.1f}% of total)')

# Create binary severity target (mirrors main notebook logic)
serious_labels = {'incapacitating injury', 'fatal'}
crashes_geo['severity_category'] = crashes_geo['most_severe_injury'].apply(
    lambda x: 1 if x in serious_labels else 0
)

print('\nSeverity distribution:')
print(crashes_geo['severity_category'].value_counts(normalize=True).round(4))

Records with coordinates: 430,749 (99.1% of total)

Severity distribution:
severity_category
0    0.9816
1    0.0184
Name: proportion, dtype: float64


In [5]:
# Filter to Chicago bounding box to remove any data entry outliers
CHI_LAT = (41.6, 42.1)
CHI_LON = (-87.95, -87.5)

crashes_geo = crashes_geo[
    crashes_geo['latitude'].between(*CHI_LAT) &
    crashes_geo['longitude'].between(*CHI_LON)
].copy()

print(f'Records within Chicago bounds: {len(crashes_geo):,}')

Records within Chicago bounds: 430,725


## 3. Map 1 — Crash Density Heatmap

This map shows where crashes are most concentrated across Chicago. Hotspots tend to cluster along major arterials and downtown corridors.

In [ ]:
# Sample for performance (heatmap works well with ~50k points)
heatmap_sample = crashes_geo.sample(n=min(50_000, len(crashes_geo)), random_state=42)

heat_data = heatmap_sample[['latitude', 'longitude']].values.tolist()

# Build map
m_heat = folium.Map(
    location=[41.8781, -87.6298],  # Chicago center
    zoom_start=11,
    tiles='CartoDB positron'
)

HeatMap(
    heat_data,
    radius=8,
    blur=10,
    max_zoom=13,
    gradient={0.2: '#ffffb2', 0.4: '#fecc5c', 0.6: '#fd8d3c',
              0.8: '#f03b20', 1.0: '#bd0026'}
).add_to(m_heat)

# Title overlay
title_html = '''
<div style="position: fixed; top: 15px; left: 50%; transform: translateX(-50%);
            background: white; padding: 10px 18px; border-radius: 8px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.25); z-index: 9999;
            font-family: Arial, sans-serif; font-size: 15px; font-weight: bold;">
    Chicago Traffic Crash Density (2021–2024)
</div>
'''
m_heat.get_root().html.add_child(folium.Element(title_html))

m_heat.save('docs/maps/heatmap.html')
print('Heatmap saved to docs/maps/heatmap.html')
m_heat

Heatmap saved to docs/maps/heatmap.html


## 4. Map 2 — Severity Point Map

Each point represents an individual crash, colored by outcome. **Red = serious injury (fatal or incapacitating)**, **Blue = non-serious**. Sampled to 6,000 points for browser performance — serious crashes are shown at full count to preserve their visibility.

In [7]:
serious = crashes_geo[crashes_geo['severity_category'] == 1].copy()
non_serious = crashes_geo[crashes_geo['severity_category'] == 0].copy()

# Fill any NaN string fields to avoid .title() errors
serious['most_severe_injury'] = serious['most_severe_injury'].fillna('unknown')
serious['prim_contributory_cause'] = serious['prim_contributory_cause'].fillna('unknown')
non_serious['most_severe_injury'] = non_serious['most_severe_injury'].fillna('unknown')

# Sample both classes to keep file size under GitHub's 100MB limit
NON_SERIOUS_SAMPLE = 3_000
SERIOUS_SAMPLE = 2_000
non_serious_sample = non_serious.sample(n=min(NON_SERIOUS_SAMPLE, len(non_serious)), random_state=42)
serious_sample = serious.sample(n=min(SERIOUS_SAMPLE, len(serious)), random_state=42)

print(f'Serious crashes plotted:     {len(serious_sample):,}')
print(f'Non-serious crashes plotted: {len(non_serious_sample):,}')

m_sev = folium.Map(
    location=[41.8781, -87.6298],
    zoom_start=11,
    tiles='CartoDB positron'
)

# Layer: non-serious (blue, plotted first so serious sits on top)
non_serious_layer = folium.FeatureGroup(name='Non-Serious Crashes', show=True)
for _, row in non_serious_sample.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color='#4393c3',
        fill=True,
        fill_color='#4393c3',
        fill_opacity=0.4,
        weight=0.5,
        popup=folium.Popup(
            f"<b>Severity:</b> Non-Serious<br>"
            f"<b>Injury:</b> {row['most_severe_injury'].title()}",
            max_width=200
        )
    ).add_to(non_serious_layer)
non_serious_layer.add_to(m_sev)

# Layer: serious (red)
serious_layer = folium.FeatureGroup(name='Serious Crashes (Fatal / Incapacitating)', show=True)
for _, row in serious_sample.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=5,
        color='#d6604d',
        fill=True,
        fill_color='#d6604d',
        fill_opacity=0.75,
        weight=1,
        popup=folium.Popup(
            f"<b>Severity:</b> Serious<br>"
            f"<b>Injury:</b> {row['most_severe_injury'].title()}<br>"
            f"<b>Cause:</b> {row['prim_contributory_cause'].title()}",
            max_width=250
        )
    ).add_to(serious_layer)
serious_layer.add_to(m_sev)

folium.LayerControl(collapsed=False).add_to(m_sev)

# Legend
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
            background: white; padding: 12px 16px; border-radius: 8px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.25); font-family: Arial, sans-serif; font-size: 13px;">
    <b>Crash Severity</b><br>
    <span style="color:#d6604d;">&#9679;</span> Serious (Fatal / Incapacitating)<br>
    <span style="color:#4393c3;">&#9679;</span> Non-Serious
</div>
'''
m_sev.get_root().html.add_child(folium.Element(legend_html))

title_html = '''
<div style="position: fixed; top: 15px; left: 50%; transform: translateX(-50%);
            background: white; padding: 10px 18px; border-radius: 8px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.25); z-index: 9999;
            font-family: Arial, sans-serif; font-size: 15px; font-weight: bold;">
    Crash Severity Map — Chicago (2021–2024)
</div>
'''
m_sev.get_root().html.add_child(folium.Element(title_html))

m_sev.save('docs/maps/severity_map.html')
print('Severity map saved to docs/maps/severity_map.html')
m_sev

Serious crashes plotted:     2,000
Non-serious crashes plotted: 3,000
Severity map saved to docs/maps/severity_map.html


In [8]:
print(crashes_geo['severity_category'].value_counts())
print(crashes_geo['most_severe_injury'].value_counts().head(10))

severity_category
0    422787
1      7938
Name: count, dtype: int64
most_severe_injury
no indication of injury     363435
nonincapacitating injury     37741
reported, not evident        20576
incapacitating injury         7402
fatal                          536
Name: count, dtype: int64


## 5. Map 3 — Serious Injury Rate by Community Area (Choropleth)

This map aggregates crash data to Chicago's 77 community areas and calculates the **serious injury rate** (serious crashes / total crashes) per neighborhood. Darker shading indicates a higher proportion of crashes resulting in serious injury — directly relevant to Vision Zero's equity focus.

In [9]:
COMMUNITY_AREAS_URL = 'https://raw.githubusercontent.com/blackmad/neighborhoods/master/chicago.geojson'

try:
    community_areas = gpd.read_file(COMMUNITY_AREAS_URL)
    print(f'Downloaded {len(community_areas)} community area boundaries.')
    print(community_areas.columns.tolist())
except Exception as e:
    print(f'Could not download boundaries: {e}')

Downloaded 98 community area boundaries.
['name', 'cartodb_id', 'created_at', 'updated_at', 'geometry']


In [10]:
# Convert crash points to GeoDataFrame
geometry = [Point(xy) for xy in zip(crashes_geo['longitude'], crashes_geo['latitude'])]
crashes_gdf = gpd.GeoDataFrame(crashes_geo, geometry=geometry, crs='EPSG:4326')

# Ensure CRS matches
community_areas = community_areas.to_crs('EPSG:4326')

# Spatial join: assign each crash to a community area
crashes_joined = gpd.sjoin(
    crashes_gdf,
    community_areas[['name', 'geometry']],
    how='left',
    predicate='within'
)

print(f'Crashes assigned to a community area: '
      f'{crashes_joined["name"].notna().sum():,}')

Crashes assigned to a community area: 430,539


In [11]:
# Aggregate: serious injury rate per community area
area_stats = (
    crashes_joined
    .groupby('name')
    .agg(
        total_crashes=('severity_category', 'count'),
        serious_crashes=('severity_category', 'sum')
    )
    .reset_index()
)
area_stats['serious_rate'] = (
    area_stats['serious_crashes'] / area_stats['total_crashes'] * 100
).round(2)

print('Top 10 community areas by serious injury rate:')
area_stats.sort_values('serious_rate', ascending=False).head(10)

Top 10 community areas by serious injury rate:


,name,total_crashes,serious_crashes,serious_rate
68,Oakland,817,32,3.92
15,Burnside,376,13,3.46
7,Avalon Park,2750,86,3.13
21,Douglas,2740,79,2.88
88,Washington Park,3066,88,2.87
31,Garfield Park,10582,301,2.84
27,Englewood,12365,325,2.63
84,Ukrainian Village,991,26,2.62
43,Jackson Park,1699,44,2.59
35,Grand Crossing,9334,242,2.59


In [12]:
# Drop timestamp columns that break JSON serialization
community_areas = community_areas.drop(columns=['created_at', 'updated_at'], errors='ignore')

# Merge stats back to GeoDataFrame
community_areas_merged = community_areas.merge(
    area_stats,
    on='name',
    how='left'
)

# Build choropleth
m_choro = folium.Map(
    location=[41.8781, -87.6298],
    zoom_start=11,
    tiles='CartoDB positron'
)

folium.Choropleth(
    geo_data=community_areas_merged.__geo_interface__,
    name='Serious Injury Rate (%)',
    data=community_areas_merged,
    columns=['name', 'serious_rate'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.4,
    legend_name='Serious Injury Rate (% of crashes)',
    nan_fill_color='lightgray',
    highlight=True
).add_to(m_choro)

# Tooltip on hover
folium.GeoJson(
    community_areas_merged.__geo_interface__,
    style_function=lambda x: {'fillOpacity': 0, 'weight': 0},
    tooltip=folium.GeoJsonTooltip(
        fields=['name', 'total_crashes', 'serious_crashes', 'serious_rate'],
        aliases=['Neighborhood:', 'Total Crashes:', 'Serious Crashes:', 'Serious Injury Rate (%):']
    )
).add_to(m_choro)

folium.LayerControl().add_to(m_choro)

title_html = '''
<div style="position: fixed; top: 15px; left: 50%; transform: translateX(-50%);
            background: white; padding: 10px 18px; border-radius: 8px;
            box-shadow: 0 2px 8px rgba(0,0,0,0.25); z-index: 9999;
            font-family: Arial, sans-serif; font-size: 15px; font-weight: bold;">
    Serious Injury Rate by Neighborhood — Chicago (2021–2024)
</div>
'''
m_choro.get_root().html.add_child(folium.Element(title_html))

m_choro.save('docs/maps/choropleth.html')
print('Choropleth saved to docs/maps/choropleth.html')
m_choro

Choropleth saved to docs/maps/choropleth.html


## 6. Key Spatial Findings

The maps above reveal several patterns worth highlighting:

- **Crash density** clusters along Chicago's major north-south arterials (Western, Pulaski, Cicero) and the downtown Loop, consistent with high traffic volume corridors.
- **Serious crashes** are distributed more broadly than total crashes, with notable concentrations on the South and West Sides — communities that also face documented underinvestment in traffic infrastructure.
- **The choropleth** exposes meaningful variation in serious injury *rates* across neighborhoods, which raw crash counts alone would obscure. Some lower-volume areas show disproportionately high rates, suggesting road design or speed environment factors beyond traffic volume.
- These geographic patterns reinforce the equity framing from the main analysis and provide actionable spatial targets for CDOT's Vision Zero interventions.

---

**View the full analysis:** [notebook.ipynb](notebook.ipynb) | [GitHub Repo](https://github.com/ckucewicz/traffic_crash_prediction)